# Photometric quality cuts across all LSST alert splits
Each split's results are appended to disk as soon as
they're computed, so if something crashes partway through you don't lose
earlier splits (just rerun -- see the "resume" note in the loop cell).

In [1]:
import gzip
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd
from astropy.io import fits

## Paths and constants

In [3]:
PROJECT_ROOT = Path("/Users/jennakempster-taylor/Documents/bayesn-lsst/jkt-project")

lsst_zip = PROJECT_ROOT / "data" / "LSST_ALERTS.zip"

OUTPUT_CSV = PROJECT_ROOT / "data" / "all_alert_candidates_photcuts.csv"

DETECTION_BIT = 1
GARBAGE_BIT = 64
VALID_BANDS = np.array(list("ugrizy"))

SELECTION = {
    # tweak
    "min_clean": 10,          # minimum clean photometric measurements
    "min_detections": 8,
    "min_bands": 2,
    "min_nights": 6,
    "min_time_span_days": 8,
}

## Core functions (unchanged from your existing notebook)

In [7]:
def read_gzipped_fits_table(archive: ZipFile, member_name: str):
    """Helper for gzip files"""
    raw = gzip.decompress(archive.read(member_name))
    with fits.open(BytesIO(raw), memmap=False) as hdul:
        data = hdul[1].data.copy()
        columns = list(hdul[1].columns.names)
    return data, columns


def clean_snid(value) -> str:
    """Return SNID as python string"""
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="replace").strip()
    return str(value).strip()


def get_lightcurve(head_row, phot_table):
    """Extract phot rows for one object using SNANA pointers"""
    start = int(head_row["PTROBS_MIN"]) - 1
    stop = int(head_row["PTROBS_MAX"])
    expected = int(head_row["NOBS"])
    lc = phot_table[start:stop]
    if len(lc) != expected:
        snid = clean_snid(head_row["SNID"])
        raise ValueError(
            f"SNID {snid}: expected {expected} rows, "
            f"but pointer range returned {len(lc)}."
        )
    return lc

def photometry_mask(lc, *, detections_only=False):
    """Mask for one object phot
    Keeps: finite MJD, FLUXCAL, and FLUXCALERR
    FLUXCALERR > 0
    real lsst band (removes separator)
    no garbage 64 bit

    if detections_only = True, only allow detection bit = 1
    """
    mjd = np.asarray(lc["MJD"], dtype=float)
    flux = np.asarray(lc["FLUXCAL"], dtype=float)
    fluxerr = np.asarray(lc["FLUXCALERR"], dtype=float)
    band = np.char.strip(np.asarray(lc["BAND"]).astype(str))
    flag = np.asarray(lc["PHOTFLAG"], dtype=int)

    mask = (
        np.isfinite(mjd)
        & np.isfinite(flux)
        & np.isfinite(fluxerr)
        & (fluxerr > 0)
        & np.isin(band, VALID_BANDS)
        & ((flag & GARBAGE_BIT) == 0)
    )

    if detections_only:
        mask &= (flag & DETECTION_BIT) != 0

    return mask


def valid_redshift(value) -> bool:
    value = float(value)
    return np.isfinite(value) and value > 0


def summarise_object(row, phot_table):
    lc = get_lightcurve(row, phot_table)
    good = photometry_mask(lc)
    detection = photometry_mask(lc, detections_only=True)

    mjd = np.asarray(lc["MJD"], dtype=float)
    band = np.char.strip(np.asarray(lc["BAND"]).astype(str))

    clean_mjd = mjd[good]
    clean_band = band[good]

    return {
        "SNID": clean_snid(row["SNID"]),
        "SNTYPE": int(row["SNTYPE"]),
        "NOBS": int(row["NOBS"]),
        "N_CLEAN": int(good.sum()),
        "N_DETECT": int(detection.sum()),
        "N_BANDS": int(np.unique(clean_band).size),
        "N_NIGHTS": int(np.unique(np.floor(clean_mjd)).size) if clean_mjd.size else 0,
        "TIME_SPAN": float(np.ptp(clean_mjd)) if clean_mjd.size > 1 else 0.0,
        "HAS_REDSHIFT": valid_redshift(row["REDSHIFT_FINAL"]),
        "RA": float(row["RA"]),
        "DEC": float(row["DEC"]),
        "MWEBV": float(row["MWEBV"]),
        "PEAKMJD": float(row["PEAKMJD"]),
    }

## Sanity check on one split first

Before looping over all 10, confirm this still works exactly as before on
Split001 -- if column names or the zip layout have changed, better to find
out here than 8 splits into the full loop.

In [8]:
head_name = "LSST_ALERTS/LSST/LSST_CHUNK01_SPLIT001_HEAD.FITS.gz"
phot_name = "LSST_ALERTS/LSST/LSST_CHUNK01_SPLIT001_PHOT.FITS.gz"

with ZipFile(lsst_zip, "r") as archive:
    head_data, head_columns = read_gzipped_fits_table(archive, head_name)
    phot_data, phot_columns = read_gzipped_fits_table(archive, phot_name)

print(f"Head objects: {len(head_data):,}")
print(f"Phot rows: {len(phot_data):,}")

# Confirm RA/DEC/MWEBV/PEAKMJD really are top-level columns (not just
# HOSTGAL_RA etc.) before trusting them in the loop below
for col in ["RA", "DEC", "MWEBV", "PEAKMJD", "REDSHIFT_FINAL"]:
    assert col in head_columns, f"Missing expected column: {col}"
print("Column check passed.")

test_summary = pd.DataFrame(
    summarise_object(row, phot_data) for row in head_data[:200]
)
test_summary.head()

Head objects: 51,000
Phot rows: 273,660
Column check passed.


,SNID,SNTYPE,NOBS,N_CLEAN,N_DETECT,N_BANDS,N_NIGHTS,TIME_SPAN,HAS_REDSHIFT,RA,DEC,MWEBV,PEAKMJD
0,00000533-33a2-42cf-bfa4-e4951522102f,0,2,2,2,1,2,4.0419,False,282.909373,-18.680916,0.150664,0.01
1,00001b7e-822c-4030-9b16-cf9cdca02319,0,2,2,2,1,2,30.9735,False,252.565374,-18.918723,0.477121,0.01
2,00001dd3-dcbc-436f-99d8-5f310e2212ba,0,2,2,2,1,2,10.0075,False,307.694958,-23.628002,0.064505,0.01
3,00004f91-a3cc-4f46-912b-df70ebb4c131,0,2,2,2,1,2,11.9482,False,318.419696,-12.661889,0.041788,0.01
4,00005c75-321d-47a7-ad59-9abf192e251b,0,2,2,2,1,2,8.0208,False,260.692795,-16.199554,0.264681,0.01


## Loop over all 10 splits

Applies `SELECTION` and appends each split's passing candidates straight to
`all_alert_candidates_photcuts.csv` on disk. Re-running this cell will
**append duplicate rows** if a split already ran -- if you need to resume
after a crash, either delete the output CSV and start over, or manually
edit the `split_numbers` list below to skip splits already written.

In [9]:
split_numbers = [f"{i:03d}" for i in range(1, 11)]

# Start fresh -- comment this out if you're deliberately resuming and want
# to keep rows already written from a previous partial run
if OUTPUT_CSV.exists():
    OUTPUT_CSV.unlink()

total_candidates = 0

for split_num in split_numbers:
    print(f"\n--- Processing split {split_num} ---")

    head_name = f"LSST_ALERTS/LSST/LSST_CHUNK01_SPLIT{split_num}_HEAD.FITS.gz"
    phot_name = f"LSST_ALERTS/LSST/LSST_CHUNK01_SPLIT{split_num}_PHOT.FITS.gz"

    with ZipFile(lsst_zip, "r") as archive:
        head_data, _ = read_gzipped_fits_table(archive, head_name)
        phot_data, _ = read_gzipped_fits_table(archive, phot_name)

    summary = pd.DataFrame(
        summarise_object(row, phot_data) for row in head_data
    )

    candidate_mask = (
        (summary["N_CLEAN"] >= SELECTION["min_clean"])
        & (summary["N_DETECT"] >= SELECTION["min_detections"])
        & (summary["N_BANDS"] >= SELECTION["min_bands"])
        & (summary["N_NIGHTS"] >= SELECTION["min_nights"])
        & (summary["TIME_SPAN"] >= SELECTION["min_time_span_days"])
    )

    candidates = summary.loc[candidate_mask].copy()
    candidates["SPLIT"] = split_num

    n_pass = candidate_mask.sum()
    print(f"Split {split_num}: {n_pass} / {len(candidate_mask)} pass photometric cuts")

    write_header = not OUTPUT_CSV.exists()
    candidates.to_csv(OUTPUT_CSV, mode="a", header=write_header, index=False)

    total_candidates += n_pass

print(f"\nTotal candidates across all splits: {total_candidates}")
print(f"Saved to: {OUTPUT_CSV}")


--- Processing split 001 ---
Split 001: 1549 / 51000 pass photometric cuts

--- Processing split 002 ---
Split 002: 1446 / 50084 pass photometric cuts

--- Processing split 003 ---
Split 003: 1476 / 50000 pass photometric cuts

--- Processing split 004 ---
Split 004: 1489 / 50000 pass photometric cuts

--- Processing split 005 ---
Split 005: 1487 / 50000 pass photometric cuts

--- Processing split 006 ---
Split 006: 1492 / 50000 pass photometric cuts

--- Processing split 007 ---
Split 007: 1519 / 50000 pass photometric cuts

--- Processing split 008 ---
Split 008: 1501 / 50000 pass photometric cuts

--- Processing split 009 ---
Split 009: 1486 / 50000 pass photometric cuts

--- Processing split 010 ---
Split 010: 1453 / 50000 pass photometric cuts

Total candidates across all splits: 14898
Saved to: /Users/jennakempster-taylor/Documents/bayesn-lsst/jkt-project/data/all_alert_candidates_photcuts.csv


## Quick look at the combined result

In [10]:
all_candidates = pd.read_csv(OUTPUT_CSV)
print(f"Total rows: {len(all_candidates)}")
print(all_candidates["SPLIT"].value_counts().sort_index())
all_candidates.head()

Total rows: 14898
SPLIT
1     1549
2     1446
3     1476
4     1489
5     1487
6     1492
7     1519
8     1501
9     1486
10    1453
Name: count, dtype: int64


,SNID,SNTYPE,NOBS,N_CLEAN,N_DETECT,N_BANDS,N_NIGHTS,TIME_SPAN,HAS_REDSHIFT,RA,DEC,MWEBV,PEAKMJD,SPLIT
0,000d948b-2c8c-4008-9438-1aa93db9f293,0,14,14,12,4,7,61.7404,False,62.095059,-48.723228,0.011406,61057.179688,1
1,001014f4-7826-402e-acd4-65bd91aff9e8,0,20,20,20,4,8,140.3629,False,61.682099,-47.263068,0.013252,61095.058594,1
2,0012c5d3-39b4-4f7d-a089-5a6ff8d1caeb,0,23,23,14,4,8,127.3737,False,59.641589,-50.597931,0.006461,61096.109375,1
3,0013fab7-58cc-4ad3-a142-296f38b10a63,0,26,26,21,4,10,40.9355,False,62.484588,-47.013235,0.012247,61097.136719,1
4,00143d0c-fe4a-4ab3-ab88-6ad47acd757a,0,108,108,59,6,47,155.6719,False,148.790563,2.496666,0.034642,61056.187500,1
